# Differential accessibility: which regions differ between two cancer cell lines?

**The data.** ATAC-seq of two versions of the colorectal cancer line COLO320, from
[Wu et al. 2019, *Nature*](https://www.nature.com/articles/s41586-019-1763-5), two replicates each:

- **COLO320DM**: the *MYC* oncogene is amplified on **ecDNA**, small circles of DNA outside the chromosomes known to drive tumor evolution and drug resistance
- **COLO320HSR**: the same amplified DNA is **integrated into a chromosome**.

Step 2 (the previous step of this workshop) turned the reads into a table of **reads per peak per sample**. This notebook starts
from that table.

**The plan:**

1. Find the peaks that differ between the lines.
2. Check whether the differences could be technical.
3. Account for copy number.
4. Test the paper's claim that ecDNA is more accessible than the same DNA in a chromosome.
5. Find the transcription factor motifs in the peaks that differ.

Cells marked **✏️ Try it** have settings to change: edit, re-run, and see what happens.

*To run in Colab: Runtime → Change runtime type → **R**, then run the cells in order.*

## 0. Setup (about 3 minutes)

In [ ]:
# Install the R packages (prebuilt binaries from r2u, so this takes about a minute instead of 20)
needed <- c("DESeq2", "dplyr", "tidyr", "ggplot2", "patchwork")
if (!all(sapply(needed, requireNamespace, quietly = TRUE))) {
    codename <- system("lsb_release -cs", intern = TRUE)
    download.file(sprintf("https://github.com/eddelbuettel/r2u/raw/master/inst/scripts/add_cranapt_%s.sh", codename), "add_cranapt.sh")
    system("bash add_cranapt.sh > /dev/null")
    bspm::enable(); options(bspm.version.check = FALSE)
    install.packages(needed)
}
suppressPackageStartupMessages({ library(DESeq2); library(dplyr); library(ggplot2); library(patchwork) })
theme_set(theme_bw(base_size = 15))
options(repr.plot.width = 9, repr.plot.height = 5.5)

In [ ]:
# Install HOMER, for the motif analysis at the end (about a minute; no genome needed)
if (system("which findMotifs.pl", ignore.stdout = TRUE) != 0) {
    dir.create("homer", showWarnings = FALSE)
    download.file("http://homer.ucsd.edu/homer/configureHomer.pl", "homer/configureHomer.pl")
    system("cd homer && perl configureHomer.pl -install homer > install.log 2>&1")
    Sys.setenv(PATH = paste(file.path(getwd(), "homer/bin"), Sys.getenv("PATH"), sep = ":"))
}
system("which findMotifs.pl")

In [ ]:
# Download the data (one archive on Google Drive, about 12 MB) and unpack it into data/
drive_file_id <- "1oVtCZvKucNm6xm2qVrCtrRdCSp_erdUY"      # from the Drive share link: https://drive.google.com/file/d/<FILE_ID>/view
if (!file.exists("data/peaks.tsv.gz")) {
    url <- sprintf("https://drive.usercontent.google.com/download?id=%s&export=download&confirm=t", drive_file_id)
    download.file(url, "differential_atac_data.tar.gz", mode = "wb")
    untar("differential_atac_data.tar.gz")
}
list.files("data")

## 1. The data

One row per peak: where it is, a few annotations, and each sample's read count.

In [ ]:
peaks <- read.delim("data/peaks.tsv.gz")
samples <- c("COLO320DM_REP1", "COLO320DM_REP2", "COLO320HSR_REP1", "COLO320HSR_REP2")
nrow(peaks)
head(peaks[, c("chr", "start", "end", "gc", "feature", "gene", samples)])

The pipeline's quality summary. **FRiP** (fraction of reads in peaks) measures signal to noise.

In [ ]:
read.delim("data/library_qc.tsv")

**Where the data came from.** From the paper's [SRA records](https://www.ncbi.nlm.nih.gov/bioproject/PRJNA506071):

| Data | Line | Uploaded to SRA |
|---|---|---|
| ATAC-seq, 2 replicates | DM | 21 Nov 2018 |
| ATAC-seq, 2 replicates | HSR | 29 May 2019 |

The DM data went up around the paper's submission, the HSR data six months later, during revision. So it appears the "DM vs HSR" contrast is equivalent to "batch 1 vs batch 2". Just something to keep in mind for every difference we find.

## 2. Differential peaks

DESeq2 treats each peak like a gene: it normalizes for sequencing depth, estimates how much counts vary
between replicates, and tests each peak for a difference between the lines.

In [ ]:
counts <- as.matrix(peaks[, samples])
rownames(counts) <- peaks$peak
sample_info <- data.frame(line = factor(c("DM", "DM", "HSR", "HSR"), levels = c("HSR", "DM")),   # HSR is the reference
                          row.names = samples)
dds <- DESeqDataSetFromMatrix(counts, sample_info, design = ~ line)
dds <- DESeq(dds, fitType = "local", quiet = TRUE)     # about a minute

**✏️ Try it.** A peak is differential if its adjusted p-value is below `padj_cutoff` and its fold change is
large enough. Two ways to ask for "large enough":

- `test_fold_change = FALSE`: test for *any* difference, then keep peaks whose log2 fold change is above
  `lfc_cutoff`. Common, but loose.
- `test_fold_change = TRUE`: test directly whether the change is bigger than `lfc_cutoff`. Stricter, and the
  better choice with only two replicates.

How many peaks are called each way? What happens with `lfc_cutoff = 0`?

In [ ]:
padj_cutoff <- 0.05
lfc_cutoff <- 1              # log2 fold change: 1 means 2-fold
test_fold_change <- TRUE

call_peaks <- function(dds) {
    res <- results(dds, alpha = padj_cutoff, lfcThreshold = if (test_fold_change) lfc_cutoff else 0)
    call <- case_when(
        res$padj < padj_cutoff & res$log2FoldChange >  lfc_cutoff ~ "up in DM",
        res$padj < padj_cutoff & res$log2FoldChange < -lfc_cutoff ~ "up in HSR",
        TRUE ~ "not significant")
    data.frame(peak = rownames(res), log2FC = res$log2FoldChange, padj = res$padj, call = call)
}
naive <- call_peaks(dds)
table(naive$call)

In [ ]:
call_colors <- c("not significant" = "gray75", "up in DM" = "firebrick", "up in HSR" = "steelblue")

ggplot(filter(naive, !is.na(padj)), aes(log2FC, -log10(padj), color = call)) +
    geom_point(size = 0.3, alpha = 0.5) +
    scale_color_manual(values = call_colors) +
    guides(color = guide_legend(override.aes = list(size = 3, alpha = 1))) +
    labs(x = "log2 fold change (DM / HSR)", y = "-log10 adjusted p-value", color = NULL,
         title = "Differential peaks")

Thousands of peaks differ, more of them "up in HSR". But the really extreme ones are "up in DM". Before interpreting any of it: could the differences be
technical?

## 3. Before believing a difference

Many things besides biology change read counts, and with 100,000 peaks even small technical effects come out
as significant. The usual suspects:

| Bias | What it does | Here |
|---|---|---|
| **Depth** | More reads overall, more in every peak | DESeq2's size factors correct it |
| **Signal-to-noise** | In a noisier library (lower FRiP), peaks stand out less | MA plot, below |
| **GC content** | PCR copies some GC levels better than others, differently in each batch | GC plot, below |
| **Mappability** | Repeats get few reads | the same in all four libraries (same read length and genome), so it cancels |
| **Copy number** | More copies of DNA, more reads | section 4 |
| **Batch** | Many small differences between prep days | can't be separated from the lines here |

### MA plot: does the fold change depend on peak strength?

One point per peak: its strength (mean normalized count, x) against its fold change (y). Most peaks don't
change, so the trend line (the median in 10 bins of strength) should run flat along zero. A bend means
something technical, such as different signal-to-noise, affects weak and strong peaks differently.

In [ ]:
color_by_gc <- FALSE      # ✏️ Try it: set to TRUE

ma <- data.frame(mean_count = rowMeans(counts(dds, normalized = TRUE)), log2FC = naive$log2FC,
                 group = if (color_by_gc) as.character(cut(peaks$gc, c(0, 0.4, 0.5, 1),
                             labels = c("< 40% GC", "40-50% GC", "> 50% GC"))) else "all peaks") |>
    filter(!is.na(log2FC))
trend <- ma |> group_by(group) |> mutate(bin = ntile(mean_count, 10)) |>
    group_by(group, bin) |> summarize(mean_count = median(mean_count), log2FC = median(log2FC), .groups = "drop")

ggplot(ma, aes(mean_count, log2FC)) +
    geom_point(size = 0.2, alpha = 0.1, color = "gray60") +
    geom_hline(yintercept = 0, color = "gray50") +
    geom_line(data = trend, aes(color = group), linewidth = 1) +
    scale_color_manual(values = c("all peaks" = "black", "< 40% GC" = "#2166AC", "40-50% GC" = "gray50", "> 50% GC" = "#B2182B")) +
    scale_x_log10() + coord_cartesian(ylim = c(-3, 3)) +
    labs(x = "mean normalized count (peak strength)", y = "log2 fold change (DM / HSR)", color = "median of",
         title = "MA plot")

The trend bends up for the strongest peaks. With `color_by_gc = TRUE`, the lines for AT-rich and GC-rich peaks
are nearly flat, just at different heights: the strongest peaks are mostly GC-rich promoters, and the bend
comes from GC, not signal-to-noise. So look at GC directly.

### GC plot: does the fold change depend on GC content?

Median fold change of peaks in 20 bins of GC content, for promoters and for everything else (distal peaks).

In [ ]:
naive$gc <- peaks$gc
naive$type <- ifelse(peaks$feature == "promoter-TSS", "promoter", "distal")

gc_trend <- naive |>
    filter(!is.na(log2FC)) |>
    group_by(type) |> mutate(gc_bin = ntile(gc, 20)) |>
    group_by(type, gc_bin) |> summarize(gc = median(gc), median_log2FC = median(log2FC), .groups = "drop")
ggplot(gc_trend, aes(gc, median_log2FC, color = type)) +
    geom_hline(yintercept = 0, color = "gray50") + geom_line(linewidth = 1) + geom_point() +
    scale_color_manual(values = c(distal = "gray30", promoter = "darkorange")) +
    labs(x = "peak GC content", y = "median log2 fold change (DM / HSR)", color = NULL,
         title = "Fold change depends on GC")

AT-rich peaks look "up in HSR" and GC-rich peaks "up in DM", with a sharp step between about 40% and 50% GC, in
promoters and distal peaks alike. With one batch per line, is that a PCR difference between the batches?

### Is it PCR?

PCR acts on every DNA fragment according to its own GC, whether it came from open chromatin or not. So a PCR
bias would also show up **between peaks**. The next plot makes the same DM vs HSR comparison fragment by
fragment: for fragments in peaks, and for nucleosome-sized fragments (180–250 bp) from the background, which come
from closed chromatin but went through the same PCR. (Computed from the aligned reads, in regions where the two
lines have equal copy number.)

In [ ]:
frag <- read.delim("data/fragment_gc.tsv.gz") |>
    filter(location == "peak" | (location == "background" & length >= 180 & length < 250)) |>
    mutate(fragments = ifelse(location == "peak", "in peaks", "background, nucleosome-sized"),
           line = ifelse(grepl("DM", sample), "DM", "HSR")) |>
    group_by(fragments, line, gc) |> summarize(n = sum(n), .groups = "drop") |>
    group_by(fragments, line) |> mutate(share = n / sum(n)) |> ungroup() |>          # each line's share of fragments at each GC
    tidyr::pivot_wider(names_from = line, values_from = c(n, share)) |>
    filter(n_DM >= 500, n_HSR >= 500)                                                   # enough fragments for a stable ratio

ggplot(frag, aes(gc, log2(share_DM / share_HSR), color = fragments)) +
    geom_hline(yintercept = 0, color = "gray50") + geom_line(linewidth = 1) +
    scale_color_manual(values = c("in peaks" = "darkorange", "background, nucleosome-sized" = "gray30")) +
    labs(x = "fragment GC", y = "log2 DM / HSR\n(share of fragments)", color = NULL, title = "Is it PCR?")

Fragments in peaks show the step. Background fragments are flat where nearly all fragments are (the ends of the
line have few fragments). **It isn't PCR:** the difference is only where the chromatin is open.

### An open question

AT-rich open sites are more accessible in HSR, and GC-rich ones in DM. It isn't a sequencing artifact, and the
same pattern appears in an independent single-cell ATAC dataset of these two lines
([Hung et al. 2021](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE160148)). So it looks like real
biology, but its cause is unknown.

When a GC trend *is* technical, tools such as `cqn` remove it. Here that would erase a real difference, so this
notebook doesn't correct for GC and keeps the pattern in mind instead. **What would you check next?**

## 4. Copy number

A region with more copies gives more reads without being more accessible. Each line's copy number here comes from
whole-genome sequencing (WGS) in the same paper. (Reads between peaks can also estimate it, when there is no WGS.)

Below, in 1 Mb windows along the genome: each line's copy number (relative to its genome median), the difference
between them, and the median fold change of the peaks in each window. All on a log2 scale, blue = lower, red =
higher; gray = too few peaks.

In [ ]:
cn <- read.delim("data/copy_number_1mb.tsv.gz")
peak_fc_1mb <- naive |>
    mutate(chr = peaks$chr, start = floor(peaks$start / 1e6) * 1e6) |>
    group_by(chr, start) |> summarize(median_log2FC = median(log2FC, na.rm = TRUE), n = n(), .groups = "drop") |>
    filter(n >= 5)
tracks <- left_join(cn, peak_fc_1mb, by = c("chr", "start")) |>
    transmute(chr = factor(sub("chr", "", chr), levels = c(1:22, "X")), start,
              "DM copy number" = log2_DM, "HSR copy number" = log2_HSR,
              "DM - HSR copy number" = log2_DM - log2_HSR, "peak fold change\n(DM / HSR)" = median_log2FC) |>
    tidyr::pivot_longer(-c(chr, start), names_to = "track", values_to = "log2") |>
    mutate(track = factor(track, levels = unique(track)))

options(repr.plot.width = 18, repr.plot.height = 5)
ggplot(tracks, aes(start / 1e6 + 0.5, 1, fill = log2)) +
    geom_tile(width = 1, height = 1) +
    facet_grid(track ~ chr, scales = "free_x", space = "free_x", switch = "y") +
    scale_fill_gradient2(low = "#2166AC", mid = "white", high = "#B2182B", midpoint = 0, limits = c(-1.5, 1.5),
                         oob = scales::squish, na.value = "gray85") +
    labs(x = NULL, y = NULL, fill = "log2") +
    theme(axis.text = element_blank(), axis.ticks = element_blank(), panel.grid = element_blank(),
          panel.spacing = unit(0.1, "lines"), strip.text.y.left = element_text(angle = 0), strip.text.x = element_text(size = 11))
options(repr.plot.width = 9, repr.plot.height = 5.5)

Where DM has more copies (red in the third track), its peaks look "more accessible" (red in the fourth). The most
extreme case is the *MYC* amplicon.

**✏️ Try it.** How much of the strongest signal comes from amplicons, which hold under 1% of peaks? Change `top_n`.

In [ ]:
top_n <- 100
peaks |>
    mutate(in_top = rank(-rowMeans(counts), ties.method = "first") <= top_n) |>
    group_by(region_class) |>
    summarize(n_in_top = sum(in_top), percent_of_top = round(100 * sum(in_top) / top_n, 1),
              percent_of_all_peaks = round(100 * n() / nrow(peaks), 2)) |>
    mutate(enrichment = round(percent_of_top / percent_of_all_peaks, 1)) |>
    arrange(desc(n_in_top))

The chr8 part of the *MYC* amplicon: each line's copy number (top, from WGS) and each peak's fold change (bottom).

In [ ]:
segments <- read.delim("data/amplicon_segments.tsv")
genes <- read.delim("data/myc_region_genes.tsv")
myc_peaks <- naive |> mutate(chr = peaks$chr, start = peaks$start) |> filter(chr == "chr8", start > 126.3e6, start < 128.2e6)
region <- c(126.3, 128.2)       # Mb

# copy number as a connected step line: each segment's start and end, in order along the chromosome
steps <- segments |> arrange(line, start) |>
    tidyr::pivot_longer(c(start, end), values_to = "position") |> arrange(line, position)

# genes: put each gene in the first row where it doesn't overlap the gene (or label) before it
genes <- genes |> mutate(from = pmax(start, region[1] * 1e6) / 1e6, to = pmin(end, region[2] * 1e6) / 1e6) |> arrange(from)
row_end <- c(); genes$row <- NA
for (i in seq_len(nrow(genes))) {
    label_end <- genes$from[i] + 0.02 * nchar(genes$name[i])       # rough width of the label, in Mb
    r <- which(row_end < genes$from[i])[1]
    if (is.na(r)) r <- length(row_end) + 1
    genes$row[i] <- r; row_end[r] <- max(genes$to[i], label_end) + 0.03
}

cn_panel <- ggplot(steps, aes(position / 1e6, copy_number, color = line)) + geom_path(linewidth = 1.2) +
    scale_y_log10() + scale_color_manual(values = c(DM = "firebrick", HSR = "steelblue")) +
    labs(x = NULL, y = "copy number\n(WGS)", color = NULL, title = "The MYC amplicon") +
    theme(axis.text.x = element_blank())
fc_panel <- ggplot(myc_peaks, aes(start / 1e6, log2FC, color = call)) + geom_point() +
    scale_color_manual(values = call_colors) +
    labs(x = NULL, y = "log2 fold change\n(DM / HSR)", color = NULL) +
    theme(axis.text.x = element_blank())
gene_panel <- ggplot(genes) +
    geom_segment(aes(x = from, xend = to, y = -row, yend = -row, color = type), linewidth = 2) +
    geom_text(aes(x = from, y = -row, label = name), size = 3.5, hjust = 0, vjust = -0.8, fontface = "italic") +
    scale_color_manual(values = c(protein_coding = "black", lncRNA = "gray60")) +
    scale_y_continuous(expand = expansion(add = c(0.4, 0.9))) +
    labs(x = "chr8 (Mb)", y = NULL, color = NULL) +
    theme(axis.text.y = element_blank(), axis.ticks.y = element_blank(), panel.grid = element_blank())

options(repr.plot.width = 12, repr.plot.height = 8)
(cn_panel / fc_panel / gene_panel) + plot_layout(heights = c(2, 2, 1.3)) & coord_cartesian(xlim = region)
options(repr.plot.width = 9, repr.plot.height = 5.5)

Where HSR has far fewer copies (the dips in its line), DM's peaks are "up" 10-fold or more: roughly the
difference in copy number.

**Add copy number to the model.** Give DESeq2 a normalization factor for every peak and sample: the sample's size
factor times its line's copy number at that peak. Twice the copies, twice the expected reads.

In [ ]:
copy_number <- cbind(peaks$copy_ratio_DM, peaks$copy_ratio_DM, peaks$copy_ratio_HSR, peaks$copy_ratio_HSR)
has_cn <- complete.cases(copy_number)

dds_cn <- DESeqDataSetFromMatrix(counts[has_cn, ], sample_info, design = ~ line)
factors <- matrix(sizeFactors(dds), nrow(counts), 4, byrow = TRUE)[has_cn, ] * copy_number[has_cn, ]
normalizationFactors(dds_cn) <- factors / exp(rowMeans(log(factors)))
dds_cn <- DESeq(dds_cn, fitType = "local", quiet = TRUE)

cn_corrected <- call_peaks(dds_cn)
cn_corrected$region_class <- peaks$region_class[has_cn]

rbind(data.frame(analysis = "standard", region_class = peaks$region_class, call = naive$call),
      data.frame(analysis = "with copy number", region_class = cn_corrected$region_class, call = cn_corrected$call)) |>
    count(analysis, region_class, call) |>
    tidyr::pivot_wider(names_from = call, values_from = n, values_fill = 0) |>
    arrange(region_class, analysis)

Almost all of the amplicon's "differential accessibility" was copy number. What remains is mostly off the
amplicons, where copy number is equal.

The volcano plot again, before and after adding copy number:

In [ ]:
volcano <- bind_rows("standard" = select(naive, log2FC, padj, call),
                     "with copy number" = select(cn_corrected, log2FC, padj, call), .id = "analysis") |>
    filter(!is.na(padj))
options(repr.plot.width = 14, repr.plot.height = 5.5)
ggplot(volcano, aes(log2FC, -log10(padj), color = call)) +
    geom_point(size = 0.3, alpha = 0.5) + facet_wrap(~ analysis) +
    scale_color_manual(values = call_colors) +
    guides(color = guide_legend(override.aes = list(size = 3, alpha = 1))) +
    labs(x = "log2 fold change (DM / HSR)", y = "-log10 adjusted p-value", color = NULL)
options(repr.plot.width = 9, repr.plot.height = 5.5)

With copy number in the model, the tower of amplicon peaks is gone, and the two sides look much alike. The few
points still far up are amplicon peaks with thousands to tens of thousands of reads, where copy number doesn't
explain everything. With that many reads, even a modest difference is extremely significant: a p-value measures
how sure we are, not how big the difference is. Section 5 asks whether ecDNA is more accessible per copy.

## 5. Is ecDNA more accessible?

The original paper this data came from asked a different question: **per copy of DNA**, is the ecDNA amplicon more accessible than the same
DNA in HSR's chromosome? Their measure: ATAC reads in 10 kb windows, divided by the WGS copy number, relative to
a typical window elsewhere in the genome.

In [ ]:
bins <- read.delim("data/bins_10kb.tsv.gz")
atac <- bins[, grep("_peaks_", names(bins))]           # ATAC reads in peaks + between peaks
atac_total <- sapply(c("DM_REP1", "DM_REP2", "HSR_REP1", "HSR_REP2"), function(s) rowSums(atac[, grep(s, names(atac))]))
atac_cpm <- sweep(atac_total, 2, colSums(atac_total), "/") * 1e6

per_copy <- data.frame(
    where = ifelse(bins$myc_amplicon, "MYC amplicon", "rest of genome"),
    DM  = rowMeans(atac_cpm[, 1:2]) / (bins$wgs_DM  / median(bins$wgs_DM)),
    HSR = rowMeans(atac_cpm[, 3:4]) / (bins$wgs_HSR / median(bins$wgs_HSR)))
per_copy$DM  <- per_copy$DM  / median(per_copy$DM[per_copy$where == "rest of genome"])
per_copy$HSR <- per_copy$HSR / median(per_copy$HSR[per_copy$where == "rest of genome"])

per_copy |> group_by(where) |> summarize(windows = n(), DM = median(DM), HSR = median(HSR))
per_copy |> tidyr::pivot_longer(c(DM, HSR), names_to = "line", values_to = "per_copy") |>
    filter(per_copy > 0) |>
    ggplot(aes(where, per_copy, fill = line)) + geom_boxplot(outlier.size = 0.3) + scale_y_log10() +
    scale_fill_manual(values = c(DM = "firebrick", HSR = "steelblue")) +
    labs(x = NULL, y = "ATAC reads per copy of DNA\n(relative to a typical window)", fill = NULL)

## 6. What is left, and where

Where are the peaks that still differ once copy number is in the model, and how GC-rich are they?

In [ ]:
left <- cn_corrected |> mutate(feature = peaks$feature[has_cn], gc = peaks$gc[has_cn])

feature_colors <- c("promoter-TSS" = "#2a78d6", "exon" = "#eb6834", "intron" = "#1baf7a", "TTS" = "#eda100", "Intergenic" = "#e87ba4")
left |> count(call, feature) |> group_by(call) |> mutate(fraction = n / sum(n)) |>
    mutate(feature = factor(feature, levels = names(feature_colors))) |>
    ggplot(aes(call, fraction, fill = feature)) + geom_col(color = "white", linewidth = 0.6, width = 0.7) +
    geom_text(aes(label = ifelse(fraction >= 0.05, paste0(round(100 * fraction), "%"), "")),
              position = position_stack(vjust = 0.5), size = 4, color = "gray15") +
    scale_fill_manual(values = feature_colors) +
    labs(x = NULL, y = "fraction of peaks", fill = "genomic feature", title = "Where the peaks are")
left |> group_by(call) |> summarize(peaks = n(), median_gc = round(median(gc), 2), percent_below_40_GC = round(100 * mean(gc < 0.4)))

Peaks up in HSR are rarely promoters and mostly AT-rich: the GC pattern from section 3. Peaks up in DM look like
unchanged peaks in GC content.

### Look at them in IGV

**Always look at your data.** A list of peaks is only numbers. In a genome browser you can see whether a
"differential" peak really looks different in both replicates, or is a spike of a few reads, a repeat, or the edge
of an amplicon.

The next cell saves each set as a BED file (red = up in DM, blue = up in HSR), and every peak's results as a
table (`differential_peaks.tsv`) to sort and filter yourself, for example in Excel. Download them (in Colab: the folder
icon on the left, then right-click → Download) and drag them into the IGV session from Step 2 (`igv_local`), which
has each line's coverage.

In [ ]:
diff_peaks <- cn_corrected |>
    mutate(chr = peaks$chr[has_cn], start = peaks$start[has_cn], end = peaks$end[has_cn], gene = peaks$gene[has_cn])
for (set in c("up in DM", "up in HSR")) {
    x <- filter(diff_peaks, call == set) |> arrange(chr, start)
    color <- if (set == "up in DM") "178,34,34" else "70,130,180"
    writeLines(c(sprintf('track name="%s" itemRgb="On"', set),
                 sprintf("%s\t%d\t%d\t%s|log2FC=%.1f\t0\t.\t%d\t%d\t%s", x$chr, x$start, x$end, x$gene, x$log2FC, x$start, x$end, color)),
               paste0(gsub(" ", "_", set), ".bed"))
}

# every peak's results, one row per peak
diff_peaks |>
    transmute(peak, chr, start, end, gene, feature = peaks$feature[has_cn], region_class, gc = round(peaks$gc[has_cn], 3),
              log2FC = round(log2FC, 3), padj = signif(padj, 3), call) |>
    bind_cols(as.data.frame(counts[has_cn, ])) |>                           # the four raw read counts
    write.table("differential_peaks.tsv", sep = "\t", quote = FALSE, row.names = FALSE)
list.files(pattern = "\\.bed$|differential_peaks")

**✏️ Try it.** Paste a location into IGV's search box. Does the peak look different in the DM and HSR coverage,
in both replicates? These are the most significant peaks where the two lines have the same copy number, so the
coverage heights are comparable. Try a few from each set.

In [ ]:
look_at <- "up in HSR"      # or "up in DM"

diff_peaks |>
    filter(call == look_at, region_class == "off-amplicon, CN equal") |>
    arrange(padj) |> head(10) |>
    transmute(igv_location = sprintf("%s:%d-%d", chr, start - 2000, end + 2000), gene,
              log2FC = round(log2FC, 1), padj = signif(padj, 2))

## 7. Which transcription factors? Motif enrichment with HOMER

HOMER counts how often each known TF motif occurs in a set of peaks, compared with a background: here, the background is peaks that
did *not* change. It matches the background's GC content to each set, so the GC pattern doesn't simply turn into
"AT-rich motifs up in HSR".

**✏️ Try it.** `regions`: all peaks, or only those off the amplicons with equal copy number (the cleanest comparison).

In [ ]:
regions <- "off-amplicon, CN equal"      # or "all"

seqs <- readLines("data/peaks_200bp.fa.gz")
sequences <- setNames(seqs[c(FALSE, TRUE)], sub("^>", "", seqs[c(TRUE, FALSE)]))

results_now <- cn_corrected
if (regions != "all") results_now <- filter(results_now, region_class == regions)

write_fasta <- function(ids, file) writeLines(paste0(">", ids, "\n", sequences[ids]), file)
write_fasta(results_now$peak[results_now$call == "up in DM"], "up_in_DM.fa")
write_fasta(results_now$peak[results_now$call == "up in HSR"], "up_in_HSR.fa")
unchanged <- results_now$peak[results_now$call == "not significant" & !is.na(results_now$padj) & results_now$padj > 0.5]
set.seed(1)
write_fasta(sample(unchanged, min(20000, length(unchanged))), "background.fa")    # 20,000 unchanged peaks are plenty, and much faster
table(results_now$call)

In [ ]:
# about 5 minutes for both sets
for (set in c("up_in_DM", "up_in_HSR")) {
    system(sprintf("findMotifs.pl %s.fa fasta homer_%s -fasta background.fa -nomotif -mset vertebrates -p 2 > homer_%s.log 2>&1", set, set, set))
}
read_homer <- function(set) {
    x <- read.delim(sprintf("homer_%s/knownResults.txt", set), check.names = FALSE)
    data.frame(set = set, motif = sub("/.*", "", x[[1]]), log10_p = -x[["Log P-value"]] / log(10),
               q_value = x[["q-value (Benjamini)"]], percent_targets = x[[7]], percent_background = x[[9]])
}
motifs <- rbind(read_homer("up_in_DM"), read_homer("up_in_HSR"))

top <- motifs |> group_by(set) |> slice_max(log10_p, n = 10) |> pull(motif) |> unique()
ggplot(filter(motifs, motif %in% top), aes(set, factor(motif, levels = rev(top)), fill = pmin(log10_p, 30))) +
    geom_tile() + geom_text(aes(label = ifelse(q_value < 0.05, "*", "")), size = 5) +
    scale_fill_gradient(low = "white", high = "firebrick", name = "-log10 p") +
    labs(x = NULL, y = NULL, title = "Motifs enriched in differential peaks (* q < 0.05)")

Peaks up in DM carry TCF/LEF motifs (the TFs of Wnt signaling, central in colorectal cancer), homeobox and RUNX
motifs. Peaks up in HSR carry AP-1 (FOS/JUN) and E-box motifs (MITF, USF).

**✏️ Try it.** A motif **logo** shows the DNA a TF binds: one stack of letters per position, taller letters for
positions that matter more. Show the top motifs of one set, then of the other. Or set `search` to a TF name (for
example `"CTCF"`, `"LEF1"`, `"AP-1"`, `"CDX"`) to see where its motif ranks in each set. (HOMER draws logos for
its top 60 motifs; `homer_up_in_DM/knownResults.html` has the full report.)

In [ ]:
show_set <- "up_in_DM"       # or "up_in_HSR"
show_top <- 5
search <- ""                 # e.g. "CTCF": show where matching motifs rank instead of the top ones

known <- read.delim(sprintf("homer_%s/knownResults.txt", show_set), check.names = FALSE)
rows <- if (search == "") seq_len(show_top) else head(grep(search, known[[1]], ignore.case = TRUE), show_top)
for (i in rows) {
    q <- known[i, "q-value (Benjamini)"]
    IRdisplay::display_markdown(sprintf("**#%d %s**: in %s of %s peaks vs %s of background (q %s)", i,
        sub("/.*", "", known[i, 1]), known[i, 7], show_set, known[i, 9], if (q == 0) "< 0.0001" else paste("=", signif(q, 2))))
    logo <- sprintf("homer_%s/knownResults/known%d.logo.svg", show_set, i)
    if (file.exists(logo)) IRdisplay::display_svg(file = logo) else IRdisplay::display_markdown("(no logo: HOMER draws only its top 60)")
}

## 8. What can we conclude?

- **Check before you interpret.** An MA plot, a GC plot and fold changes along the genome catch most problems.
- **Copy number** explains nearly all of the amplicon's differences. Put it in the model.
- **GC** is one of the most common biases seen in sequencing data, but it can be technical or biological (or both) so think carefully about this one!
- **Per copy, the ecDNA amplicon appears more accessible** than the same DNA in a chromosome, reproducing the analysis and findings of the paper that generated this data.
- **Different TF programs:** Wnt (TCF/LEF), homeobox and RUNX motifs up in DM; AP-1 and E-box motifs up in HSR.
- **Caveat:** each line was made in its own batch, so any single difference could partly be batch.